# 4주차. 정적 웹 크롤링과 텍스트 분석

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

이 노트북은 수업 주차 흐름을 참고해 우리 방식으로 재구성한 실행형 설명 자료입니다. 외부 서비스 호출 없이 실행되도록 작은 샘플 데이터를 사용합니다.

## 정적 HTML 파싱
정적 웹페이지는 응답 HTML 안에 필요한 정보가 들어 있습니다. 표준 라이브러리의 HTMLParser로도 핵심 구조를 추출할 수 있습니다.

In [1]:
from html.parser import HTMLParser
import pandas as pd
from collections import Counter

html = '''
<article class="book" data-rating="5"><h3>Python Web</h3><p class="price">31.5</p></article>
<article class="book" data-rating="4"><h3>Data App</h3><p class="price">28.0</p></article>
<article class="book" data-rating="3"><h3>Clean Code Lab</h3><p class="price">22.0</p></article>
'''

class BookParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows = []
        self.current = None
        self.field = None

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "article":
            self.current = {"rating": int(attrs["data-rating"])}
        elif self.current is not None and tag == "h3":
            self.field = "title"
        elif self.current is not None and tag == "p":
            self.field = attrs.get("class")

    def handle_data(self, data):
        text = data.strip()
        if self.current is not None and self.field and text:
            self.current[self.field] = text

    def handle_endtag(self, tag):
        if tag == "article" and self.current is not None:
            self.rows.append(self.current)
            self.current = None
        self.field = None

parser = BookParser()
parser.feed(html)
books = pd.DataFrame(parser.rows)
books["price"] = books["price"].astype(float)
display(books)

,rating,title,price
0,5,Python Web,31.5
1,4,Data App,28.0
2,3,Clean Code Lab,22.0


## 간단한 텍스트 벡터화
CountVectorizer의 기본 아이디어는 문서를 토큰으로 나누고 단어 등장 횟수를 세는 것입니다.

In [2]:
documents = books["title"].str.lower().tolist()
vocabulary = sorted({token for doc in documents for token in doc.split()})
matrix = []
for doc in documents:
    counts = Counter(doc.split())
    matrix.append([counts[token] for token in vocabulary])

vector_table = pd.DataFrame(matrix, columns=vocabulary)
assert vector_table.shape[0] == len(books)
display(vector_table)

,app,clean,code,data,lab,python,web
0,0,0,0,0,0,1,1
1,1,0,0,1,0,0,0
2,0,1,1,0,1,0,0
